In [10]:
import pandas as pd
from loguru import logger
import os

In [4]:
COLS_TO_DROP = [
    # --- Negligible numerical ---
    "mths_since_last_delinq",
    "mths_since_last_record",
    "total_acc",
    "collections_12_mths_ex_med",
    "mths_since_last_major_derog",
    "acc_now_delinq",
    "tot_coll_amt",
    "mths_since_rcnt_il",
    "total_bal_il",
    "max_bal_bc",
    "total_cu_tl",
    "mo_sin_old_il_acct",
    "num_il_tl",
    "pct_tl_nvr_dlq",
    # --- Unanalysed / low-info numerical ---
    "chargeoff_within_12_mths",
    "delinq_amnt",
    "mo_sin_rcnt_rev_tl_op",
    "mo_sin_rcnt_tl",
    "mths_since_recent_bc",
    "mths_since_recent_inq",
    "mths_since_recent_revol_delinq",
    "num_actv_bc_tl",
    "num_bc_sats",
    "num_bc_tl",
    "num_sats",
    "num_tl_120dpd_2m",
    "num_tl_30dpd",
    "num_tl_90g_dpd_24m",
    "num_tl_op_past_12m",
    "tax_liens",
    "total_bal_ex_mort",
    "total_il_high_credit_limit",
    "bc_open_to_buy",
    # --- Joint application columns ---
    "application_type",           # after filtering Individual
    "annual_inc_joint",
    "dti_joint",
    "verification_status_joint",
    "revol_bal_joint",
    "sec_app_earliest_cr_line",
    "sec_app_inq_last_6mths",
    "sec_app_mort_acc",
    "sec_app_open_acc",
    "sec_app_revol_util",
    "sec_app_open_act_il",
    "sec_app_num_rev_accts",
    "sec_app_chargeoff_within_12_mths",
    "sec_app_collections_12_mths_ex_med",
    "sec_app_mths_since_last_major_derog",
    # --- Categorical drops ---
    "emp_title",                  # 300k+ unique values
]

In [5]:
file_path = '../data/raw/loan_preprocessed.parquet'

def read_data(file_path = file_path):
    """Reads the dataset from the preprocessed data - Parquet file."""
    logger.info(f"Reading data from {file_path}...")
    try: 
        df = pd.read_parquet(file_path)
        logger.success("Data read successfully.")
        return df
    except Exception as e:
        logger.error(f"Error reading data: {e}")
        raise e 


df = read_data()
df.sample(10)
# rename target column for use of helper functions




2026-03-09 11:49:46.130 | INFO     | __main__:read_data:5 - Reading data from ../data/raw/loan_preprocessed.parquet...
2026-03-09 11:49:46.665 | SUCCESS  | __main__:read_data:8 - Data read successfully.


,loan_amnt,term,emp_title,emp_length,home_ownership,annual_inc,verification_status,loan_status,purpose,addr_state,...,sec_app_earliest_cr_line,sec_app_inq_last_6mths,sec_app_mort_acc,sec_app_open_acc,sec_app_revol_util,sec_app_open_act_il,sec_app_num_rev_accts,sec_app_chargeoff_within_12_mths,sec_app_collections_12_mths_ex_med,sec_app_mths_since_last_major_derog
442659,5000,36 months,Managet,< 1 year,MORTGAGE,35000.0,Source Verified,Fully Paid,other,PA,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1272846,16000,36 months,driver,9 years,MORTGAGE,100000.0,Verified,Fully Paid,house,CA,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
403923,16000,36 months,None,None,MORTGAGE,94000.0,Not Verified,Fully Paid,home_improvement,FL,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1171526,30000,36 months,Credit Collections Manager,2 years,RENT,110000.0,Verified,Fully Paid,debt_consolidation,CA,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
734503,35000,36 months,Owner,10+ years,MORTGAGE,250000.0,Source Verified,Fully Paid,credit_card,GA,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
272298,20000,36 months,Manager,10+ years,MORTGAGE,80000.0,Verified,Fully Paid,debt_consolidation,WA,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
649046,20000,36 months,Consultant Nurse,3 years,MORTGAGE,91000.0,Source Verified,Charged Off,debt_consolidation,NC,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
150782,14400,60 months,Corp acct Admin,10+ years,MORTGAGE,40000.0,Not Verified,Fully Paid,home_improvement,CA,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
889337,5550,36 months,allegiant home care,2 years,RENT,27000.0,Not Verified,Fully Paid,debt_consolidation,NY,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
172503,13500,36 months,Repair Service Specialist,6 years,RENT,50000.0,Not Verified,Fully Paid,credit_card,NY,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
df.loan_status

0           Fully Paid
1           Fully Paid
2           Fully Paid
3           Fully Paid
4           Fully Paid
              ...     
1303633    Charged Off
1303634    Charged Off
1303635     Fully Paid
1303636     Fully Paid
1303637     Fully Paid
Name: loan_status, Length: 1303638, dtype: object

In [ ]:

df_new= df.drop(columns=COLS_TO_DROP, errors='ignore')

df_new.rename(columns={'loan_status': 'target'}, inplace=True)

df_new['target'] = df_new['target'].map({
    'Fully Paid': 0,
    'Charged Off': 1,
    'Default': 1    

})

,loan_amnt,term,emp_length,home_ownership,annual_inc,verification_status,loan_status,purpose,addr_state,dti,...,mo_sin_old_rev_tl_op,mort_acc,num_actv_rev_tl,num_op_rev_tl,num_rev_accts,num_rev_tl_bal_gt_0,percent_bc_gt_75,pub_rec_bankruptcies,tot_hi_cred_lim,total_bc_limit
180670,6000,36 months,10+ years,MORTGAGE,39000.0,Not Verified,Fully Paid,car,CA,9.14,...,237.0,1.0,3.0,4.0,19.0,3.0,0.0,1.0,25463.0,14500.0
62885,1000,36 months,10+ years,RENT,65000.0,Verified,Fully Paid,other,FL,28.49,...,255.0,3.0,5.0,7.0,8.0,5.0,20.0,0.0,95288.0,53300.0
217022,16000,60 months,5 years,MORTGAGE,50000.0,Not Verified,Fully Paid,debt_consolidation,WA,18.75,...,124.0,1.0,5.0,7.0,11.0,5.0,20.0,0.0,217091.0,35900.0
63101,13500,60 months,10+ years,RENT,62000.0,Verified,Fully Paid,debt_consolidation,TX,18.66,...,180.0,0.0,4.0,9.0,12.0,4.0,20.0,0.0,57890.0,12100.0
750175,18000,60 months,10+ years,RENT,52000.0,Verified,Charged Off,debt_consolidation,FL,33.28,...,125.0,0.0,6.0,8.0,8.0,6.0,50.0,0.0,60464.0,20800.0
759244,7000,36 months,None,OWN,30000.0,Source Verified,Fully Paid,debt_consolidation,IN,14.84,...,122.0,0.0,3.0,6.0,14.0,3.0,66.7,1.0,14800.0,8700.0
963875,9600,36 months,5 years,MORTGAGE,63000.0,Not Verified,Fully Paid,home_improvement,WI,8.42,...,310.0,1.0,2.0,3.0,9.0,2.0,0.0,0.0,111821.0,800.0
1134294,7000,36 months,8 years,MORTGAGE,55000.0,Not Verified,Fully Paid,debt_consolidation,VA,10.69,...,190.0,4.0,3.0,5.0,7.0,3.0,100.0,0.0,151400.0,5000.0
267812,24000,60 months,1 year,MORTGAGE,110000.0,Source Verified,Charged Off,debt_consolidation,IL,6.44,...,342.0,11.0,6.0,6.0,10.0,6.0,0.0,1.0,94100.0,16600.0
956124,4000,36 months,6 years,RENT,50000.0,Not Verified,Fully Paid,credit_card,CA,2.26,...,150.0,0.0,2.0,7.0,9.0,2.0,33.3,0.0,19300.0,16400.0


In [8]:
df_new.target

AttributeError: 'DataFrame' object has no attribute 'target'

In [14]:
df_new.columns

Index(['loan_amnt', 'term', 'emp_length', 'home_ownership', 'annual_inc',
       'verification_status', 'loan_status', 'purpose', 'addr_state', 'dti',
       'delinq_2yrs', 'earliest_cr_line', 'inq_last_6mths', 'open_acc',
       'pub_rec', 'revol_bal', 'revol_util', 'tot_cur_bal', 'open_acc_6m',
       'open_act_il', 'open_il_12m', 'open_il_24m', 'il_util', 'open_rv_12m',
       'open_rv_24m', 'all_util', 'total_rev_hi_lim', 'inq_fi', 'inq_last_12m',
       'acc_open_past_24mths', 'avg_cur_bal', 'bc_util',
       'mo_sin_old_rev_tl_op', 'mort_acc', 'num_actv_rev_tl', 'num_op_rev_tl',
       'num_rev_accts', 'num_rev_tl_bal_gt_0', 'percent_bc_gt_75',
       'pub_rec_bankruptcies', 'tot_hi_cred_lim', 'total_bc_limit'],
      dtype='object')

In [16]:
OUTPUT_DIR = "../data/processed"
df_new.to_parquet(f"{OUTPUT_DIR}/loan_selected.parquet", index=False)
